# H&M Personalized Fashion Recommendations: Two-Tower Retrieval

This notebook demonstrates the core PyTorch architecture powering candidate retrieval for H&M fashion recommendations.

In an enterprise "retrieve-and-rank" pattern:
1. **PyTorch Two-Tower Model**: Processes user features and fashion attributes to retrieve the top candidate items in sub-5ms.
2. **Gemini & LanceDB**: Performs fine-grained semantic search and generates human-readable explanations.

### Towers Architecture:
* **User Tower**: Maps customer attributes (e.g., age, postal code/group, membership status, and historical purchase trends) into a shared $D$-dimensional space.
* **Item Tower**: Maps fashion article features (e.g., product type, garment group, color, index group, and index name) into that same space.


In [ ]:
import os
import sys
import torch
from torch.utils.data import DataLoader

# Ensure the src directory is accessible
sys.path.append(os.path.abspath('.'))

## 1. H&M Data Simulation & Ingestion

We simulate customer interaction logs and demographic data inspired by the Kaggle **H&M Personalized Fashion Recommendations** dataset.

A temporal split is applied to evaluate the model on future interactions based on historical patterns, replicating production-grade validation.


In [ ]:
from demo_recommendation_engine.pytorch_model.training.data import (
    generate_synthetic_data,
    temporal_train_val_test_split,
    RecommendationDataset
)

# Generate synthetic interactions
interactions = generate_synthetic_data(num_users=500, num_items=200, num_interactions=20000, seed=42)
train_data, val_data, test_data = temporal_train_val_test_split(interactions)

print(f"Temporal Split -> Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

# Create PyTorch DataLoaders
train_loader = DataLoader(RecommendationDataset(train_data), batch_size=256, shuffle=True, drop_last=True)
val_loader = DataLoader(RecommendationDataset(val_data), batch_size=256, shuffle=False)
test_loader = DataLoader(RecommendationDataset(test_data), batch_size=256, shuffle=False)

## 2. Two-Tower (Dual-Encoder) Architecture

The model maps disparate feature spaces into a single shared embedding space where dot-product similarity makes physical sense:

```text
[Customer Demographics] --\
                           +---> [ User Tower MLP ] ---> [User Vector] (dim=64)
[Session / Click History] -/                                      |
                                                           (Dot Product)
[Fashion Categories]    --\
                           +---> [ Item Tower MLP ] ---> [Item Vector] (dim=64)
[Item Text / Colors]      -/
```

### Addressing the Cold-Start Problem (New Users)
- **ID-Only Lookup**: Fails for new customers because no embedding exists for their ID.
- **PyTorch Demographics MLP**: Resolves this by mapping unknown IDs to a reserved `<UNKNOWN_USER>` embedding and relying on demographics (age, postal group, etc.) fed into the User Tower MLP. The network maps these attributes to the embedding space, generating relevant recommendations instantly.


In [ ]:
from demo_recommendation_engine.pytorch_model.training.model import TwoTowerModel

num_users, num_items = 500, 200
num_user_features, num_item_features = 8, 16

model = TwoTowerModel(
    num_users=num_users,
    num_items=num_items,
    embedding_dim=64,
    hidden_dim=128,
    num_user_features=num_user_features,
    num_item_features=num_item_features,
    temperature=0.07
)

print(model)

## 3. Training via In-Batch Negative Sampling

To scale to H&M's massive catalog, we train the model using **in-batch negative sampling**.

The batch forms a similarity matrix where the diagonal represents positive interactions, and all off-diagonal items act as implicit negative examples. This avoids the cost of negative sampling loops.


In [ ]:
from demo_recommendation_engine.pytorch_model.training.train import train_model, compute_metrics

all_item_ids = torch.arange(num_items)

# Run a fast training loop
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_items=num_items,
    all_item_ids=all_item_ids,
    epochs=5, # Reduced for quick demonstration
    lr=1e-3
)

## 4. Evaluation: Retrieval & Ranking Metrics

Model performance is evaluated across the catalog using information retrieval metrics:
* **Hit Rate@K**: Check if the user's clicked item appears in the top-K recommendations.
* **NDCG@K**: Measures the ranking quality of recommended fashion articles.
* **Mean Reciprocal Rank (MRR)**: Measures how quickly the first relevant item is retrieved.


In [ ]:
import json
test_metrics = compute_metrics(model, test_loader, num_items, all_item_ids, k_values=[10, 50])

print("Final Test Metrics:")
print(json.dumps(test_metrics, indent=2))

## 5. Deployment Optimization: Dynamic Quantization

To support high-throughput, low-latency CPU serving:
* **Post-Training Dynamic Quantization** converts the linear layer weights to `int8`.
* This reduces the model size by **~42%** and ensures sub-1ms CPU inference latency.


In [ ]:
import torch.nn as nn

# Measure Original Size
torch.save(model.state_dict(), "/tmp/model_original.pt")
orig_size = os.path.getsize("/tmp/model_original.pt")

# Apply Dynamic Quantization
quantized_model = torch.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
)

# Measure Quantized Size
torch.save(quantized_model.state_dict(), "/tmp/model_quantized.pt")
quant_size = os.path.getsize("/tmp/model_quantized.pt")

reduction = 100 * (1 - quant_size / orig_size)
print(f"Original Model Size:  {orig_size / 1024:.2f} KB")
print(f"Quantized Model Size: {quant_size / 1024:.2f} KB")
print(f"Storage Reduction:    {reduction:.1f}%")

### Next Steps

The trained model is optimized for local CPU serving and deployed to **Vertex AI Endpoints**. Upstream, retraining is triggered dynamically by MLOps quality gates monitoring feature and prediction drift.
